# Experiment: does a run from an EMPTY folder reproduce the published numbers? Part 2: score and compare (CPU)

`03_score` with the same two settings as part 1: ground truth is built again from LiDAR, the 7 scenes are scored
again. The last cell then reads BOTH folders, the new one and the published run's, and compares them three ways:
the predictions themselves (does the GPU give the same depth twice?), the ground truth (it must be identical), and
every number in the result files. About 5 minutes on a CPU high-RAM server.

In [1]:
# --- 1. Configuration ---
PERSIST_MODE = "drive"
DRIVE_ROOT = "/content/drive/MyDrive/vggt-omega-aura-benchmark-rerun-check"   # an EMPTY folder: nothing saved, so everything is recomputed.
# Work already saved there is skipped. To run EVERYTHING again from the images up, name an empty folder here,
# the same one in every notebook of the run. The Hugging Face token is still found in the usual folder's .env.
RUN_TAG = "phase7_front_medium"       # the folder of this run in persistent storage. A name from the time of the work:
                                      # every notebook of the run must use the same one, the results live under it
CAMERA = "front_medium"
MODELS = ["vggt_omega_512", "vggt_1b"]
BLOCKS = [("val", 12)]                # the smallest block: 7 scenes. Scored in the published run, so there is something to compare with
LIDAR_POLICY = "ouster_only"          # same six sensors in every scene, so ground-truth density is comparable
VALIDATE_DOWNLOAD = True              # run the dataset toolkit's own validator on each new block
REVERSE = False                       # forwards here; `03_score_second_server` walks the same list backwards,
                                      # so two CPU servers can share the work. This notebook alone does everything.
WORKERS = None                        # scenes scored at once. None = one per CPU core (max 8). The numbers do not depend on it

In [ ]:
# === CODE SYNC (auto-generated by `python -m vggt_aura.sync`, do not edit) ===
raise RuntimeError("The sync cell is empty. On your own machine, in the project folder, run:  python -m vggt_aura.sync   and reopen this notebook.")

In [3]:
# --- 3. Start the session ---
from vggt_aura.session import start_session

# build_cpp=True compiles the C++ geometry core on this server (about 15 s). Ground truth is then built
# with it, which gives exactly the same result as the Python reference, faster. If the build fails,
# everything still runs, in Python.
session = start_session(persist_mode=PERSIST_MODE, drive_root=DRIVE_ROOT, require_gpu=False, build_cpp=True)

Mounted at /content/drive
installing vggt_omega
installing pybind11
installing fzi_aura
persist root: /content/drive/MyDrive/vggt-omega-aura-benchmark-rerun-check
data root   : /content/data/fzi-aura (runtime disk, wiped at session end)
provenance  : provenance.jsonl | code 19ba183bdad7 | data 3404bd6b8fcd
$ cmake -S /content/vggt-omega-aura-benchmark/cpp -B /content/vggt-omega-aura-benchmark/cpp/build -DCMAKE_BUILD_TYPE=Release -Dpybind11_DIR=/usr/local/lib/python3.13/dist-packages/pybind11/share/cmake/pybind11 -DPython_EXECUTABLE=/usr/bin/python3
$ cmake --build /content/vggt-omega-aura-benchmark/cpp/build --config Release -j
C++ core    : built


In [4]:
# --- 4. Process the blocks ---
import pandas as pd
from vggt_aura import aura_data as ad, pipeline as pl

pd.set_option("display.width", 220)
chunks, scene_blocks, hub_files = ad.fetch_release_tables(session.data_root / "_release_tables")
EXCLUDED = ad.fetch_excluded_scene_ids(session.data_root / "_release_tables")   # faulty scenes the maintainers exclude
print("scenes excluded by the dataset:", len(EXCLUDED))
available = ad.available_blocks(chunks, scene_blocks, hub_files, [pl.CAMERA_LAYER, pl.LIDAR_LAYER])
import uuid
ME = ("backward-" if REVERSE else "forward-") + uuid.uuid4().hex[:6]      # this server's name on its claims
summaries, left_to_the_other = [], []
for split, block in (list(reversed(BLOCKS)) if REVERSE else list(BLOCKS)):
    if not pl.block_is_done(session.persist_root, RUN_TAG, MODELS, split, block) \
            and not pl.claim_block(session.persist_root, RUN_TAG, split, block, ME, max_age_s=1500):
        print(f"=== {pl.block_tag(split, block)}: the other server is on it, skipped ===")
        left_to_the_other.append((split, block))
        continue
    assert (split, block) in available.index, f"block {(split, block)} is not downloadable with camera + LiDAR"
    print(f"=== {pl.block_tag(split, block)} ({available.loc[(split, block), 'total_gb']} GB) ===")
    try:
        summary = pl.process_block(session, split, block, ad.block_scene_ids(scene_blocks, split, block, EXCLUDED), CAMERA, MODELS,
                                   RUN_TAG, lidar_policy=LIDAR_POLICY, validate=VALIDATE_DOWNLOAD,
                                   scene_names=ad.block_scene_names(scene_blocks, split, block, EXCLUDED), workers=WORKERS)
    finally:                 # a claim must not outlive a crash: a re-run gets a new name and would wait for it
        pl.release_claim(session.persist_root, RUN_TAG, split, block, ME)
    print(" ", summary)
    summaries.append(summary)
print()
print(pd.DataFrame([{k: v for k, v in s.items() if k not in ("sensors", "validation")} for s in summaries]).to_string(index=False))
still_open = [b for b in left_to_the_other if not pl.block_is_done(session.persist_root, RUN_TAG, MODELS, *b)]
if still_open:
    print()
    print("left to the other server and not finished yet:", still_open, "| if that server stopped, run this notebook again")

scenes excluded by the dataset: 8
=== val_block000012 (2.67 GB) ===
  downloading with the toolkit, decompressing with xz on all 8 cores


  fast unpack: {'archives': 3, 'xz_decompressed_on_all_cores': 1, 'download_s': 7.4, 'verify_and_decompress_s': 29.5, 'extract_s': 10.3}
  predictions: 0 made now, the rest loaded (0 s) | scoring 7 scenes with 7 worker(s)
  2025-06-13-07-09-37|75      30.7 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2025-06-11-12-27-55|206     33.4 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2025-06-13-07-09-37|74      31.3 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2025-06-13-07-09-37|73      27.3 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2025-06-11-14-31-00|34      50.5 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2025-06-11-14-31-00|37      45.3 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2025-06-16-12-35-26|19      41.2 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  validator: {'ok': True, 'scenes_checked': 7, 'errors': []}
  {

In [5]:
# --- 5. What the run holds so far ---
for model in MODELS:
    rows, scenes = pl.load_run(session.persist_root, RUN_TAG, model)
    print(f"{model}: {scenes['scene_id'].nunique() if len(scenes) else 0} scenes in "
          f"{scenes[['split', 'block']].drop_duplicates().shape[0] if len(scenes) else 0} blocks")

vggt_omega_512: 7 scenes in 1 blocks
vggt_1b: 7 scenes in 1 blocks


In [6]:
# --- 6. Compare with the published run: predictions, ground truth, every result ---
import numpy as np
from pathlib import Path
from vggt_aura import ground_truth as gtm, inference as inf

PUBLISHED_ROOT = Path("/content/drive/MyDrive/vggt-omega-aura-benchmark")
assert PUBLISHED_ROOT != session.persist_root, "DRIVE_ROOT still points at the published run: nothing was recomputed"
split, block = BLOCKS[0]
names = ad.block_scene_names(scene_blocks, split, block, EXCLUDED)

print("=== 1. predictions: the same images through the same weights, twice ===")
for model in MODELS:
    folder = pl.ModelRunner(model).folder
    worst_depth, worst_pose, identical = 0.0, 0.0, 0
    for name in names:
        old, _ = inf.load_predictions(*inf.prediction_paths(PUBLISHED_ROOT, name, CAMERA, folder))
        new, _ = inf.load_predictions(*inf.prediction_paths(session.persist_root, name, CAMERA, folder))
        assert np.array_equal(old["timestamps_ns"], new["timestamps_ns"]), f"{name}: different frames"
        relative = np.abs(new["depth"].astype(np.float64) - old["depth"]) / old["depth"]
        worst_depth = max(worst_depth, float(np.median(relative)))
        worst_pose = max(worst_pose, float(np.abs(new["extrinsics"] - old["extrinsics"]).max()))
        identical += bool(np.array_equal(new["depth"], old["depth"]))
    print(f"  {model:15} bit-identical depth in {identical} of {len(names)} scenes | largest per-scene MEDIAN relative depth "
          f"difference {worst_depth:.2e} | largest difference in any pose entry {worst_pose:.2e}")

print()
print("=== 2. ground truth: built twice from the same LiDAR files, must be identical ===")
for model in MODELS:
    height, width = (int(v) for v in inf.load_predictions(*inf.prediction_paths(session.persist_root, names[0], CAMERA, pl.ModelRunner(model).folder))[0]["input_hw"])
    same = 0
    for name in names:
        old, _, _ = gtm.load_scene_truth(*gtm.truth_paths(PUBLISHED_ROOT, name, CAMERA, gtm.truth_tag(width, height)))
        new, _, _ = gtm.load_scene_truth(*gtm.truth_paths(session.persist_root, name, CAMERA, gtm.truth_tag(width, height)))
        same += all(np.array_equal(a[key], b[key]) for a, b in zip(old, new) for key in ("u", "v", "depth_m", "semantic_id"))
    print(f"  {model:15} {width}x{height}: identical in {same} of {len(names)} scenes")

print()
print("=== 3. results: every number of the block's result files ===")
for model in MODELS:
    tables = []
    for root in (PUBLISHED_ROOT, session.persist_root):
        rows_path, scenes_path = pl.block_result_paths(root, RUN_TAG, model, split, block)
        tables.append((pd.read_csv(rows_path), pd.read_csv(scenes_path)))
    (rows_old, scenes_old), (rows_new, scenes_new) = tables
    keys = ["scene_id", "protocol", "stratum_type", "stratum"]
    both = rows_old.merge(rows_new, on=keys, suffixes=("_published", "_rerun"))
    assert len(both) == len(rows_old) == len(rows_new), "the two runs do not have the same rows"
    print(f"  {model}: {len(both)} rows compared | pixel counts identical: {bool((both['n_pixels_published'] == both['n_pixels_rerun']).all())}")
    for column in ("abs_rel", "delta125", "rmse_m"):
        difference = (both[f"{column}_rerun"] - both[f"{column}_published"]).abs()
        print(f"      {column:9} largest difference in any row {difference.max():.2e}")
    primary = both[(both["protocol"] == "sequence_scale") & (both["stratum_type"] == "all")]
    print("      scene AbsRel, published then re-run:")
    for _, row in primary.iterrows():
        print(f"        {row['scene_id']:28} {row['abs_rel_published']:.4f}  {row['abs_rel_rerun']:.4f}")
    print(f"      block mean AbsRel: published {primary['abs_rel_published'].mean():.4f} | re-run {primary['abs_rel_rerun'].mean():.4f}")
    pose = scenes_old.merge(scenes_new, on="scene_id", suffixes=("_published", "_rerun"))
    for column in ("rotation_deg_median", "translation_deg_median", "ate_scale_only_pct_of_path", "fx_rel_err_median"):
        difference = (pose[f"{column}_rerun"] - pose[f"{column}_published"]).abs()
        print(f"      {column:28} largest difference {difference.max():.2e}")

=== 1. predictions: the same images through the same weights, twice ===
  vggt_omega_512  bit-identical depth in 7 of 7 scenes | largest per-scene MEDIAN relative depth difference 0.00e+00 | largest difference in any pose entry 0.00e+00
  vggt_1b         bit-identical depth in 7 of 7 scenes | largest per-scene MEDIAN relative depth difference 0.00e+00 | largest difference in any pose entry 0.00e+00

=== 2. ground truth: built twice from the same LiDAR files, must be identical ===
  vggt_omega_512  640x400: identical in 7 of 7 scenes
  vggt_1b         518x322: identical in 7 of 7 scenes

=== 3. results: every number of the block's result files ===
  vggt_omega_512: 580 rows compared | pixel counts identical: True
      abs_rel   largest difference in any row 0.00e+00
      delta125  largest difference in any row 0.00e+00
      rmse_m    largest difference in any row 0.00e+00
      scene AbsRel, published then re-run:
        2025-06-13-07-09-37|75       0.0696  0.0696
        2025-06-11